In [0]:


import requests
import time
from pyspark.sql.functions import col


BOT_TOKEN = "8924265842:AAGa1JAItFbsZjd9zmDqcmp8spKb_3LTgE0"

CHAT_ID   = "@Shoghla1"

SILVER_TABLE = "depi_project.philo_files.silver_linkedin_indeed"
SENT_TABLE   = "depi_project.philo_files.telegram_sent_jobs"

TELEGRAM_API = f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage"

MAX_MESSAGES_PER_RUN = 200  # safety cap so a huge new batch doesn't spam / hit rate limits
RETRY_COUNT = 3
RETRY_DELAY_SECONDS = 3

# ---------- 1) Make sure the "sent jobs" tracking table exists ----------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SENT_TABLE} (
    url STRING,
    sent_at TIMESTAMP
) USING delta
""")

# ---------- 2) Find jobs that have not been sent yet ----------
df_jobs = spark.table(SILVER_TABLE)
df_sent = spark.table(SENT_TABLE).select("url")

new_jobs_df = df_jobs.join(df_sent, on="url", how="left_anti").limit(MAX_MESSAGES_PER_RUN)
new_jobs = new_jobs_df.collect()

print(f"New jobs to send: {len(new_jobs)}")

# ---------- 3) Helper: send a single Telegram message with retries ----------
def send_message(text: str) -> bool:
    for attempt in range(1, RETRY_COUNT + 1):
        try:
            resp = requests.post(
                TELEGRAM_API,
                data={
                    "chat_id": CHAT_ID,
                    "text": text,
                    "parse_mode": "HTML",
                    "disable_web_page_preview": False,
                },
                timeout=15,
            )
            if resp.status_code == 200:
                return True

            # Telegram rate limit: back off and retry
            if resp.status_code == 429:
                retry_after = resp.json().get("parameters", {}).get("retry_after", RETRY_DELAY_SECONDS)
                print(f"Rate limited, waiting {retry_after}s...")
                time.sleep(retry_after)
                continue

            print(f"Telegram API error ({resp.status_code}): {resp.text}")
            return False

        except requests.RequestException as e:
            print(f"Request failed (attempt {attempt}/{RETRY_COUNT}): {e}")
            time.sleep(RETRY_DELAY_SECONDS)

    return False

# ---------- 4) Build and send a message per job ----------
def escape_html(value: str) -> str:
    # Minimal escaping so job titles/company names with <, >, & don't break Telegram HTML parsing
    if value is None:
        return ""
    return str(value).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

sent_rows = []
failed_count = 0

for row in new_jobs:
    text = (
        f"💼 <b>{escape_html(row['title'])}</b>\n"
        f"━━━━━━━━━━━━━━━\n"
        f"🏢 <b>Company:</b> {escape_html(row['company'])}\n"
        f"📍 <b>Location:</b> {escape_html(row['location'])}\n"
        f"📅 <b>Date:</b> {escape_html(row['date'])}\n"
        f"🌐 <b>Source:</b> {escape_html(row['source'])}\n"
        f"\n"
        f"🔗 <a href='{row['url']}'>Apply / View job</a>"
    )

    if send_message(text):
        sent_rows.append((row["url"],))
    else:
        failed_count += 1

    time.sleep(1)  # stay well under Telegram's rate limits

# ---------- 5) Record what was successfully sent ----------
if sent_rows:
    from pyspark.sql.functions import current_timestamp, lit

    new_sent_df = (
        spark.createDataFrame(sent_rows, ["url"])
        .withColumn("sent_at", current_timestamp())
    )
    new_sent_df.write.format("delta").mode("append").saveAsTable(SENT_TABLE)

print(f"Done. Sent: {len(sent_rows)}, Failed: {failed_count}")

# Fail the notebook/job run if everything failed, so the Databricks Job shows a red X
# and you get notified instead of silently missing a day.
if new_jobs and not sent_rows:
    raise Exception("All Telegram sends failed for this run.")


New jobs to send: 187
Rate limited, waiting 31s...
Rate limited, waiting 31s...
